# Silver Layer Parquet Analysis Notebook

This notebook reads and analyzes Parquet files from the Silver layer of the data lake.
- **Source Data**: Reddit posts and La Silla Vacía articles (Bronze layer)
- **Processing**: Pandas preprocessing with data cleaning and validation (Silver layer)
- **Output**: Parquet files with traceability metadata

## Objectives:
1. Load Parquet files using Pandas
2. Perform exploratory data analysis (EDA)
3. Validate data quality
4. Verify traceability to source files
5. Generate summary statistics

In [11]:
# Import Required Libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("Libraries imported successfully")

Libraries imported successfully


## 1. Define File Paths

Set up paths to locate Parquet files in the Silver layer

In [12]:
# Define base paths
WORKSHOP_PATH = Path("..")
SILVER_PATH = WORKSHOP_PATH / "datalake_silver"
BRONZE_PATH = WORKSHOP_PATH / "datalake_bronze"

# Display paths
print("Data Lake Paths:")
print(f"  Silver (Input):  {SILVER_PATH.resolve()}")
print(f"  Bronze (Source): {BRONZE_PATH.resolve()}")

# List available Parquet files
if SILVER_PATH.exists():
    parquet_files = sorted(SILVER_PATH.glob("*.parquet"))
    print(f"\n Found {len(parquet_files)} Parquet files in Silver layer:")
    for pf in parquet_files:
        size_mb = pf.stat().st_size / (1024**2)
        print(f"  • {pf.name:50} ({size_mb:8.2f} MB)")
else:
    print(f" Silver path does not exist: {SILVER_PATH.resolve()}")

Data Lake Paths:
  Silver (Input):  /home/sbuitrago/dataAnalysis/workshop2/datalake_silver
  Bronze (Source): /home/sbuitrago/dataAnalysis/workshop2/datalake_bronze

 Found 2 Parquet files in Silver layer:
  • lasillavacia_20260409_042109.parquet               (    0.10 MB)
  • reddit_20260409_042109.parquet                     (    0.04 MB)


## 2. Read Parquet Files

Load Parquet files using Pandas and combine them for analysis

In [14]:
# Read Reddit Parquet files
print("Reading Parquet files...\n")

reddit_files = sorted(SILVER_PATH.glob("reddit_*.parquet"))
lsv_files = sorted(SILVER_PATH.glob("lasillavacia_*.parquet"))

dfs_reddit = []
dfs_lsv = []
metadata = {}

# Load Reddit data
if reddit_files:
    print(f" Reddit Parquet files: {len(reddit_files)}")
    for pf in reddit_files:
        try:
            df = pd.read_parquet(pf)
            dfs_reddit.append(df)
            print(f"   {pf.name}: {len(df)} rows × {len(df.columns)} columns")
            if df.attrs:
                metadata[pf.name] = df.attrs
        except Exception as e:
            print(f"   {pf.name}: Error - {str(e)}")
    
    # Combine all Reddit data
    if dfs_reddit:
        df_reddit = pd.concat(dfs_reddit, ignore_index=True)
        print(f"\n   Combined Reddit: {len(df_reddit)} rows × {len(df_reddit.columns)} columns\n")
else:
    print("   No Reddit Parquet files found\n")
    df_reddit = None

# Load La Silla Vacía data
if lsv_files:
    print(f" La Silla Vacía Parquet files: {len(lsv_files)}")
    for pf in lsv_files:
        try:
            df = pd.read_parquet(pf)
            dfs_lsv.append(df)
            print(f"   {pf.name}: {len(df)} rows × {len(df.columns)} columns")
            if df.attrs:
                metadata[pf.name] = df.attrs
        except Exception as e:
            print(f"   {pf.name}: Error - {str(e)}")
    
    # Combine all La Silla Vacía data
    if dfs_lsv:
        df_lsv = pd.concat(dfs_lsv, ignore_index=True)
        print(f"\n   Combined La Silla Vacía: {len(df_lsv)} rows × {len(df_lsv.columns)} columns\n")
else:
    print("  ⚠️ No La Silla Vacía Parquet files found\n")
    df_lsv = None

print(" All Parquet files loaded successfully")

Reading Parquet files...

 Reddit Parquet files: 1
   reddit_20260409_042109.parquet: 39 rows × 9 columns

   Combined Reddit: 39 rows × 9 columns

 La Silla Vacía Parquet files: 1
   lasillavacia_20260409_042109.parquet: 10 rows × 12 columns

   Combined La Silla Vacía: 10 rows × 12 columns

 All Parquet files loaded successfully


## 3. Display Data Information

Examine the structure, data types, and metadata of loaded DataFrames

In [15]:
# Reddit Data Information
if df_reddit is not None:
    print("=" * 80)
    print(" REDDIT DATA STRUCTURE")
    print("=" * 80)
    print(f"\nShape: {df_reddit.shape}")
    print(f"Memory usage: {df_reddit.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n Column Information:")
    print(df_reddit.info())
    
    print("\n Data Types:")
    print(df_reddit.dtypes)
    
    print("\n Statistical Summary:")
    print(df_reddit.describe(include='all').T)

# La Silla Vacía Data Information
if df_lsv is not None:
    print("\n" + "=" * 80)
    print(" LA SILLA VACÍA DATA STRUCTURE")
    print("=" * 80)
    print(f"\nShape: {df_lsv.shape}")
    print(f"Memory usage: {df_lsv.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    print("\n Column Information:")
    print(df_lsv.info())
    
    print("\n Data Types:")
    print(df_lsv.dtypes)
    
    print("\n Statistical Summary:")
    print(df_lsv.describe(include='all').T)

 REDDIT DATA STRUCTURE

Shape: (39, 9)
Memory usage: 0.08 MB

 Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39 entries, 0 to 38
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   title        39 non-null     object        
 1   title_clean  39 non-null     object        
 2   text         39 non-null     object        
 3   text_clean   39 non-null     object        
 4   author       39 non-null     object        
 5   date         39 non-null     datetime64[ns]
 6   score        39 non-null     int64         
 7   url          39 non-null     object        
 8   source_file  39 non-null     object        
dtypes: datetime64[ns](1), int64(1), object(7)
memory usage: 2.9+ KB
None

 Data Types:
title                  object
title_clean            object
text                   object
text_clean             object
author                 object
date           datetime64[ns]
score   

## 4. Preview Data

Display first and last rows of each dataset

In [16]:
# Reddit First Rows
if df_reddit is not None:
    print("=" * 80)
    print(" REDDIT - First 5 Rows")
    print("=" * 80)
    display(df_reddit.head())
    
    print("\n REDDIT - Last 5 Rows")
    print("=" * 80)
    display(df_reddit.tail())

# La Silla Vacía First Rows
if df_lsv is not None:
    print("\n" + "=" * 80)
    print(" LA SILLA VACÍA - First 5 Rows")
    print("=" * 80)
    display(df_lsv.head())
    
    print("\n LA SILLA VACÍA - Last 5 Rows")
    print("=" * 80)
    display(df_lsv.tail())

 REDDIT - First 5 Rows


,title,title_clean,text,text_clean,author,date,score,url,source_file
0,Colombia es muy inteligente a nivel individual...,colombia inteligente nivel individual 3stup1da...,"Aunque es un enorme DUH!, quería compartir con...",aunque enorme duh quería compartir uds esa ref...,ArquimedeanDeer,2026-04-06 21:56:17,9,https://reddit.com/r/Colombia/comments/1secx9a...,reddit_20260406_230327.json
1,"Voy a cumplir 25 años, y no he tenido experien...",voy cumplir años tenido experiencia mujeres es...,"Bueno, esa es mi ""triste"" situación. A mis cas...",bueno esa triste situación mis casi años tenid...,Party-Situation-4890,2026-04-06 20:38:59,7,https://reddit.com/r/Colombia/comments/1seaund...,reddit_20260406_230327.json
2,Cargamel le quedó en pañales a petro :v,cargamel quedó pañales petro,y falta saber las cositas que tiene con calarc...,falta saber cositas tiene calarca malboro,Any_Organization_370,2026-04-06 18:15:41,10,https://reddit.com/r/Colombia/comments/1se6t5f...,reddit_20260406_230327.json
3,El documental La ley del monte ( 1988),documental ley monte 1988,El documental La ley del monte ( 1988) muestra...,documental ley monte 1988 muestra proceso colo...,West-Juggernaut-2832,2026-04-06 17:20:32,2,https://reddit.com/r/Colombia/comments/1se57lj...,reddit_20260406_230327.json
4,lugares para comprar vestidos,lugares comprar vestidos,"Hola, alguno sabe donde puedo encontrar vestid...",hola alguno sabe puedo encontrar vestidos tipo...,Unfair-Attitude-7557,2026-04-06 16:46:48,1,https://reddit.com/r/Colombia/comments/1se49h4...,reddit_20260406_230327.json



 REDDIT - Last 5 Rows


,title,title_clean,text,text_clean,author,date,score,url,source_file
34,Me dio una crisis antes de graduarme de quimic...,dio crisis antes graduarme quimica farmaceutic...,Hola a todos! Soy nuevo por acá. Les cuento: e...,hola todos soy nuevo acá les cuento estoy nada...,ProudAd9021,2026-04-08 18:07:09,2,https://reddit.com/r/Colombia/comments/1sg0mne...,reddit_20260409_001906.json
35,La verdadera encuesta,verdadera encuesta,Ha sentido que las encuestas en Colombia no cu...,sentido encuestas colombia cuadran siendo hone...,StOchastiC_,2026-04-08 14:22:23,1,https://reddit.com/r/Colombia/comments/1sfuaoo...,reddit_20260409_001906.json
36,En sus municipios/ciudades ¿como van las campa...,municipios ciudades van campañas presidenciales,Hoy me llamo la atención que vi un afiche de F...,hoy llamo atención afiche fajardo presidente p...,GarfieldM00d,2026-04-09 01:21:52,2,https://reddit.com/r/Colombia/comments/1sgbmu2...,reddit_20260409_040255.json
37,Como conseguir empleo en CompuTrabajo,conseguir empleo computrabajo,Llevo varias semanas buscando empleo con Compu...,llevo varias semanas buscando empleo computrab...,Diobrando0312,2026-04-09 00:22:55,3,https://reddit.com/r/Colombia/comments/1sgabjq...,reddit_20260409_040255.json
38,Si alguien creará un vehículo volador completa...,alguien creará vehículo volador completamente ...,&amp;#x200B;\n\nAsí como paso con las pequeñas...,amp x200b así paso pequeñas motos eléctricas p...,Ciencia_,2026-04-09 04:06:26,1,https://reddit.com/r/Colombia/comments/1sgf5en...,reddit_20260409_040648.json



 LA SILLA VACÍA - First 5 Rows


,titulo,titulo_clean,autor,fecha,extracto,extracto_clean,contenido,contenido_clean,etiquetas,url,fuente,source_file
0,"Quilcué y Oviedo, entre la representación y la...",quilcué oviedo representación instrumentalización,Columnista invitado,2026-04-04 09:27:48-05:00,Una fórmula pone la identidad en el centro com...,fórmula pone identidad centro bandera otra sil...,"Por María Alejandra Victorino Jiménez, columni...",maría alejandra victorino jiménez columnista i...,"Opinión, Aída Quilcué, Destacado del editor, E...",https://www.lasillavacia.com/opinion/quilcue-y...,lasillavacia,lasillavacia_20260406_230327.json
1,La constitución como rehén,constitución rehén,Andrés Caro,2026-04-04 08:44:32-05:00,La constituyente le ha servido al presidente P...,constituyente servido presidente petro amenaza...,Hay una campaña que no es como las otras. Es d...,campaña otras diferente todas demás ganando to...,"Opinión, Asamblea Constituyente, Destacado del...",https://www.lasillavacia.com/opinion/la-consti...,lasillavacia,lasillavacia_20260406_230327.json
2,En Casa de Nariño juegan como los aguafiestas ...,casa nariño juegan aguafiestas triunfalismo ce...,Edgar Quintero Herrera,2026-04-06 01:00:00-05:00,,,"El 17 de marzo, el presidente Petro y el minis...",marzo presidente petro ministro armando benede...,"Nacional, Armando Benedetti, Candidatos presid...",https://www.lasillavacia.com/silla-nacional/en...,lasillavacia,lasillavacia_20260406_230327.json
3,Las relaciones secretas de los amigos de Petro...,relaciones secretas amigos petro verónica régi...,Jineth Prieto,2026-04-05 01:00:00-05:00,,,Solo dos meses después de la elección de Gusta...,solo dos meses después elección gustavo petro ...,"Nacional, Camimpeg, Cilia Flórez, Colenergy Gr...",https://www.lasillavacia.com/silla-nacional/la...,lasillavacia,lasillavacia_20260406_230327.json
4,De la Espriella y los límites de los abogados:...,espriella límites abogados esto dicen expertos,Red de Expertos,2026-04-04 00:05:00-05:00,,,Este es un espacio de debate que no compromete...,espacio debate compromete opinión silla vacía ...,"Red de Derecho Constitucional, Abelardo de la ...",https://www.lasillavacia.com/red-de-expertos/r...,lasillavacia,lasillavacia_20260406_230327.json



 LA SILLA VACÍA - Last 5 Rows


,titulo,titulo_clean,autor,fecha,extracto,extracto_clean,contenido,contenido_clean,etiquetas,url,fuente,source_file
5,Las narrativas de los candidatos presidenciale...,narrativas candidatos presidenciales primera v...,La Silla Vacía,2026-04-03 00:00:00-05:00,,,,,"abelardo de la espriella, destacado del editor...",https://www.lasillavacia.com/silla-nacional/la...,lasillavacia,lasillavacia_20260406_230327.json
6,Las fórmulas vicepresidenciales muestran que l...,fórmulas vicepresidenciales muestran política ...,Laura Camila Torrado Rangel,2026-04-02 00:05:00-05:00,,,El anuncio de las fórmulas vicepresidenciales ...,anuncio fórmulas vicepresidenciales sorprendió...,"Silla Académica, Aida Quilcué, Destacado del e...",https://www.lasillavacia.com/silla-academica/l...,lasillavacia,lasillavacia_20260406_230327.json
7,Ponderador de aprobación de Petro: la aprobaci...,ponderador aprobación petro aprobación alta 2022,Juan Manuel Pinto Hernández,2026-04-06 18:43:58-05:00,,,"En 2026, menos de dos meses antes de eleccione...",2026 menos dos meses antes elecciones aprobaci...,"Nacional, Destacado del editor, Encuestas, Gus...",https://www.lasillavacia.com/silla-nacional/po...,lasillavacia,lasillavacia_20260407_001000.json
8,Los cabos sueltos en la financiación de la cam...,cabos sueltos financiación campaña petro 2022,Carmen Carolina Garnica,2026-04-07 12:00:00-05:00,,,La grabación de la conversación que mantuvo Jo...,grabación conversación mantuvo jorge lemus ent...,"Nacional, Armando Benedetti, Campaña Petro Pre...",https://www.lasillavacia.com/silla-nacional/lo...,lasillavacia,lasillavacia_20260408_030731.json
9,Escenarios del Banco de la República para sesi...,escenarios banco república sesionar tras ruptu...,Pablo Manrique,2026-04-07 01:00:00-05:00,,,El pasado martes 31 de marzo el ministro de Ha...,pasado martes marzo ministro hacienda germán á...,"Nacional, Banco de la República, Destacado del...",https://www.lasillavacia.com/silla-nacional/es...,lasillavacia,lasillavacia_20260408_030731.json


## 5. Verify Traceability

Check traceability columns (source_file, source_path) to link Silver data back to Bronze origin

In [17]:
# Verify Traceability - Reddit
if df_reddit is not None and 'source_file' in df_reddit.columns:
    print("=" * 80)
    print(" REDDIT - TRACEABILITY VERIFICATION")
    print("=" * 80)
    
    if 'source_file' in df_reddit.columns:
        print(f"\n source_file column found")
        print(f"   Unique source files: {df_reddit['source_file'].nunique()}")
        print(f"\n   Distribution of records by source file:")
        print(df_reddit['source_file'].value_counts())
    
    if 'source_path' in df_reddit.columns:
        print(f"\n source_path column found")
        print(f"   Sample source paths:")
        for idx, path in enumerate(df_reddit['source_path'].unique()[:3]):
            print(f"     {idx + 1}. {path}")
else:
    print(" Traceability columns not found in Reddit data")

# Verify Traceability - La Silla Vacía
if df_lsv is not None and 'source_file' in df_lsv.columns:
    print("\n" + "=" * 80)
    print(" LA SILLA VACÍA - TRACEABILITY VERIFICATION")
    print("=" * 80)
    
    if 'source_file' in df_lsv.columns:
        print(f"\n source_file column found")
        print(f"   Unique source files: {df_lsv['source_file'].nunique()}")
        print(f"\n   Distribution of records by source file:")
        print(df_lsv['source_file'].value_counts())
    
    if 'source_path' in df_lsv.columns:
        print(f"\n source_path column found")
        print(f"   Sample source paths:")
        for idx, path in enumerate(df_lsv['source_path'].unique()[:3]):
            print(f"     {idx + 1}. {path}")
else:
    print("⚠️ Traceability columns not found in La Silla Vacía data")

 REDDIT - TRACEABILITY VERIFICATION

 source_file column found
   Unique source files: 8

   Distribution of records by source file:
source_file
reddit_20260408_030731.json    15
reddit_20260406_230327.json    13
reddit_20260409_001906.json     5
reddit_20260409_040255.json     2
reddit_20260407_000000.json     1
reddit_20260406_233001.json     1
reddit_20260408_032727.json     1
reddit_20260409_040648.json     1
Name: count, dtype: int64

 LA SILLA VACÍA - TRACEABILITY VERIFICATION

 source_file column found
   Unique source files: 3

   Distribution of records by source file:
source_file
lasillavacia_20260406_230327.json    7
lasillavacia_20260408_030731.json    2
lasillavacia_20260407_001000.json    1
Name: count, dtype: int64


## 6. Data Quality Analysis

Analyze null values, duplicates, and data distribution

In [19]:
# Data Quality Analysis - Reddit
if df_reddit is not None:
    print("=" * 80)
    print(" REDDIT - DATA QUALITY ANALYSIS")
    print("=" * 80)
    
    print("\n Null Values:")
    null_counts = df_reddit.isnull().sum()
    null_pct = (null_counts / len(df_reddit) * 100).round(2)
    for col, count in null_counts.items():
        if count > 0:
            print(f"  • {col}: {count} ({null_pct[col]}%)")
    
    if null_counts.sum() == 0:
        print("   No null values found!")
    
    print(f"\n Duplicate Records:")
    duplicates = df_reddit.duplicated().sum()
    print(f"  • Total duplicates: {duplicates}")
    if duplicates > 0:
        print(f"    ({duplicates / len(df_reddit) * 100:.2f}% of data)")
    else:
        print(f"   No duplicate records found!")
    
    print(f"\n Key Statistics:")
    if 'score' in df_reddit.columns:
        print(f"  • Score - Mean: {df_reddit['score'].mean():.2f}, Median: {df_reddit['score'].median():.2f}")
    if 'date' in df_reddit.columns:
        print(f"  • Date range: {df_reddit['date'].min()} to {df_reddit['date'].max()}")
    if 'author' in df_reddit.columns:
        print(f"  • Unique authors: {df_reddit['author'].nunique()}")

# Data Quality Analysis - La Silla Vacía
if df_lsv is not None:
    print("\n" + "=" * 80)
    print(" LA SILLA VACÍA - DATA QUALITY ANALYSIS")
    print("=" * 80)
    
    print("\n Null Values:")
    null_counts = df_lsv.isnull().sum()
    null_pct = (null_counts / len(df_lsv) * 100).round(2)
    for col, count in null_counts.items():
        if count > 0:
            print(f"  • {col}: {count} ({null_pct[col]}%)")
    
    if null_counts.sum() == 0:
        print("   No null values found!")
    
    print(f"\n Duplicate Records:")
    duplicates = df_lsv.duplicated().sum()
    print(f"  • Total duplicates: {duplicates}")
    if duplicates > 0:
        print(f"    ({duplicates / len(df_lsv) * 100:.2f}% of data)")
    else:
        print(f"   No duplicate records found!")
    
    print(f"\n Key Statistics:")
    if 'fecha' in df_lsv.columns:
        print(f"  • Date range: {df_lsv['fecha'].min()} to {df_lsv['fecha'].max()}")
    if 'autor' in df_lsv.columns:
        print(f"  • Unique authors: {df_lsv['autor'].nunique()}")
    if 'contenido_length' in df_lsv.columns:
        print(f"  • Content length - Mean: {df_lsv['contenido_length'].mean():.0f}, Max: {df_lsv['contenido_length'].max()}")

 REDDIT - DATA QUALITY ANALYSIS

 Null Values:
   No null values found!

 Duplicate Records:
  • Total duplicates: 0
   No duplicate records found!

 Key Statistics:
  • Score - Mean: 4.46, Median: 2.00
  • Date range: 2026-04-05 04:11:31 to 2026-04-09 04:06:26
  • Unique authors: 35

 LA SILLA VACÍA - DATA QUALITY ANALYSIS

 Null Values:
   No null values found!

 Duplicate Records:
  • Total duplicates: 0
   No duplicate records found!

 Key Statistics:
  • Date range: 2026-04-02 00:05:00-05:00 to 2026-04-07 12:00:00-05:00
  • Unique authors: 10


## 7. Summary and Conclusions

Final summary of the data analysis

In [22]:
print("=" * 80)
print(" ANALYSIS SUMMARY")
print("=" * 80)

print("\n COMPLETION STATUS:")

# Reddit Summary
if df_reddit is not None:
    print(f"\n REDDIT DATA:")
    print(f"   • Total records: {len(df_reddit):,}")
    print(f"   • Total columns: {len(df_reddit.columns)}")
    print(f"   • Memory usage: {df_reddit.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Date range: {df_reddit['date'].min() if 'date' in df_reddit.columns else 'N/A'}")
    print(f"                 {df_reddit['date'].max() if 'date' in df_reddit.columns else 'N/A'}")
    print(f"   • Data quality:  Ready for analysis")
else:
    print(f"\n REDDIT DATA:  Not available")

# La Silla Vacía Summary
if df_lsv is not None:
    print(f"\n LA SILLA VACÍA DATA:")
    print(f"   • Total records: {len(df_lsv):,}")
    print(f"   • Total columns: {len(df_lsv.columns)}")
    print(f"   • Memory usage: {df_lsv.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"   • Date range: {df_lsv['fecha'].min() if 'fecha' in df_lsv.columns else 'N/A'}")
    print(f"                 {df_lsv['fecha'].max() if 'fecha' in df_lsv.columns else 'N/A'}")
    print(f"   • Data quality:  Ready for analysis")
else:
    print(f"\n LA SILLA VACÍA DATA:  Not available")

print("\n" + "=" * 80)
print(" NOTEBOOK ANALYSIS COMPLETE!")
print("=" * 80)
print("\nNext steps:")
print("  1. Export cleaned data for further analysis")
print("  2. Create visualizations using matplotlib or plotly")
print("  3. Perform sentiment analysis on text columns")
print("  4. Generate reports for stakeholders")

 ANALYSIS SUMMARY

 COMPLETION STATUS:

 REDDIT DATA:
   • Total records: 39
   • Total columns: 9
   • Memory usage: 0.13 MB
   • Date range: 2026-04-05 04:11:31
                 2026-04-09 04:06:26
   • Data quality:  Ready for analysis

 LA SILLA VACÍA DATA:
   • Total records: 10
   • Total columns: 12
   • Memory usage: 0.42 MB
   • Date range: 2026-04-02 00:05:00-05:00
                 2026-04-07 12:00:00-05:00
   • Data quality:  Ready for analysis

 NOTEBOOK ANALYSIS COMPLETE!

Next steps:
  1. Export cleaned data for further analysis
  2. Create visualizations using matplotlib or plotly
  3. Perform sentiment analysis on text columns
  4. Generate reports for stakeholders
